# 19 LSTM与GRU - 门控循环单元

上一教程揭示了RNN的致命弱点：梯度消失导致无法学习长距离依赖。本教程将学习如何通过**门控机制**来解决这个问题。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import math

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("导入库成功！")

## 一、LSTM的核心思想

### RNN的问题回顾

普通RNN只有一个隐藏状态 $h_t$，每个时间步都用 $\tanh$ 变换。由于 $\tanh'(x) \leq 1$，多步连乘后梯度消失。

### LSTM的天才设计

LSTM引入了两个状态和三个门：

**两个状态：**
- **隐藏状态 (Hidden State)** $h_t$：输出给外界的信息
- **细胞状态 (Cell State)** $C_t$：信息高速公路，可以跨时间步无损传递

**三个门：**
1. **遗忘门 (Forget Gate)**：决定丢弃什么信息
2. **输入门 (Input Gate)**：决定存储什么新信息
3. **输出门 (Output Gate)**：决定输出什么信息

### 关键洞察

细胞状态 $C_t$ 的更新是**加法**而非乘法：
$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

这意味着梯度在 $C_t$ 上的传播可以是**恒等的**（当 $f_t \approx 1$ 时），不会被连乘衰减。
这是LSTM能解决梯度消失的根本原因！

In [ ]:
# LSTM公式详解
print("=" * 70)
print("LSTM 公式详解")
print("=" * 70)
print()

lstm_steps = """
LSTM在时间步 t 的计算：

【1. 遗忘门】决定从细胞状态中丢弃什么信息
    f_t = σ(W_f · [h_{t-1}, x_t] + b_f)
    f_t ∈ (0, 1)，0表示完全遗忘，1表示完全保留

【2. 输入门】决定在细胞状态中存储什么新信息
    i_t = σ(W_i · [h_{t-1}, x_t] + b_i)    # 决定更新哪些值
    C̃_t = tanh(W_C · [h_{t-1}, x_t] + b_C) # 候选值（新信息的候选）

【3. 更新细胞状态】
    C_t = f_t ⊙ C_{t-1} + i_t ⊙ C̃_t
    关键：这是加法！梯度可以通过加法路径无损传播

【4. 输出门】决定输出什么信息
    o_t = σ(W_o · [h_{t-1}, x_t] + b_o)
    h_t = o_t ⊙ tanh(C_t)

符号说明：
    σ = sigmoid函数 (输出范围0~1)
    ⊙ = 逐元素乘法 (Hadamard product)
    [h_{t-1}, x_t] = 拼接隐藏状态和输入
"""
print(lstm_steps)

In [ ]:
# 可视化LSTM单元结构
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('LSTM单元内部结构', fontsize=16, fontweight='bold')

# 颜色方案
sigmoid_color = '#FFD700'  # 金色 - sigmoid层
tanh_color = '#87CEEB'     # 天蓝 - tanh层
pointwise_mult = '#FF6B6B' # 红色 - 逐点乘法
pointwise_add = '#98FB98'  # 绿色 - 逐点加法
concat_color = '#DDA0DD'   # 紫色 - 拼接

# 主细胞状态线（顶部水平线）
ax.plot([1, 13], [8.5, 8.5], 'k-', linewidth=3)
ax.text(0.5, 8.5, 'C_{t-1}', fontsize=12, ha='center', va='center',
        bbox=dict(boxstyle='round', facecolor='white', edgecolor='black'))
ax.text(13.5, 8.5, 'C_t', fontsize=12, ha='center', va='center',
        bbox=dict(boxstyle='round', facecolor='white', edgecolor='black'))

# 底部隐藏状态线
ax.plot([1, 13], [1.5, 1.5], 'k-', linewidth=3)
ax.text(0.5, 1.5, 'h_{t-1}', fontsize=12, ha='center', va='center',
        bbox=dict(boxstyle='round', facecolor='white', edgecolor='black'))
ax.text(13.5, 1.5, 'h_t', fontsize=12, ha='center', va='center',
        bbox=dict(boxstyle='round', facecolor='white', edgecolor='black'))

# 输入 x_t
ax.text(7, 0.3, 'x_t', fontsize=12, ha='center', va='center',
        bbox=dict(boxstyle='round', facecolor='white', edgecolor='black'))
ax.annotate('', xy=(7, 2.5), xytext=(7, 0.7),
           arrowprops=dict(arrowstyle='->', color='black', lw=2))

# h_{t-1} 到 x_t 的拼接
ax.text(4.5, 3.0, '拼接', fontsize=10, ha='center',
        bbox=dict(boxstyle='round', facecolor=concat_color, alpha=0.7))

# 四个操作层
# 遗忘门
ax.text(2.5, 4.5, 'σ', fontsize=14, ha='center', va='center',
        bbox=dict(boxstyle='round', facecolor=sigmoid_color, alpha=0.8))
ax.text(2.5, 5.5, '遗忘门 f_t', fontsize=10, ha='center', color='black')
ax.annotate('', xy=(2.5, 8.2), xytext=(2.5, 5.8),
           arrowprops=dict(arrowstyle='->', color='black', lw=2))

# 逐点乘法（遗忘）
ax.plot([2.5, 2.5], [8.3, 8.7], 'k-', linewidth=3)
ax.text(2.5, 8.5, '×', fontsize=16, ha='center', va='center', color=pointwise_mult, fontweight='bold')

# 输入门
ax.text(5.5, 4.5, 'σ', fontsize=14, ha='center', va='center',
        bbox=dict(boxstyle='round', facecolor=sigmoid_color, alpha=0.8))
ax.text(5.5, 5.5, '输入门 i_t', fontsize=10, ha='center', color='black')

# 候选值
ax.text(8.5, 4.5, 'tanh', fontsize=14, ha='center', va='center',
        bbox=dict(boxstyle='round', facecolor=tanh_color, alpha=0.8))
ax.text(8.5, 5.5, '候选值 C̃_t', fontsize=10, ha='center', color='black')

# 逐点乘法（输入）
ax.text(7, 7, '×', fontsize=16, ha='center', va='center', color=pointwise_mult, fontweight='bold')

# 逐点加法（细胞状态更新）
ax.text(9.5, 8.5, '+', fontsize=16, ha='center', va='center', color=pointwise_add, fontweight='bold')

# 输出门
ax.text(11.5, 4.5, 'σ', fontsize=14, ha='center', va='center',
        bbox=dict(boxstyle='round', facecolor=sigmoid_color, alpha=0.8))
ax.text(11.5, 5.5, '输出门 o_t', fontsize=10, ha='center', color='black')

# tanh(C_t)
ax.text(11.5, 3, 'tanh', fontsize=14, ha='center', va='center',
        bbox=dict(boxstyle='round', facecolor=tanh_color, alpha=0.8))

# 逐点乘法（输出）
ax.text(11.5, 2, '×', fontsize=16, ha='center', va='center', color=pointwise_mult, fontweight='bold')

# 图例
legend_y = 9.5
ax.text(0.5, legend_y, '图例:', fontsize=10, fontweight='bold')
ax.add_patch(plt.Rectangle((1.5, legend_y-0.2), 0.5, 0.3, facecolor=sigmoid_color, alpha=0.8))
ax.text(2.2, legend_y, 'Sigmoid层', fontsize=9)
ax.add_patch(plt.Rectangle((4, legend_y-0.2), 0.5, 0.3, facecolor=tanh_color, alpha=0.8))
ax.text(4.7, legend_y, 'Tanh层', fontsize=9)
ax.add_patch(plt.Rectangle((6, legend_y-0.2), 0.5, 0.3, facecolor=pointwise_mult, alpha=0.8))
ax.text(6.7, legend_y, '逐元素乘法 ×', fontsize=9)
ax.add_patch(plt.Rectangle((9, legend_y-0.2), 0.5, 0.3, facecolor=pointwise_add, alpha=0.8))
ax.text(9.7, legend_y, '逐元素加法 +', fontsize=9)

plt.tight_layout()
plt.show()

print("上图展示了LSTM的核心结构：")
print("  - 顶部的水平线 C 是细胞状态，信息可以直接跨时间步传递")
print("  - 三个Sigmoid门（遗忘、输入、输出）控制信息流")
print("  - 关键：C_t = f_t × C_{t-1} + i_t × C̃_t (加法保护梯度)")

## 二、LSTM前向传播数值示例

让我们用一个简化的LSTM（去掉拼接，简化维度）来演示每一步的数值计算。

In [ ]:
# LSTM数值示例 - 简化版
print("=" * 70)
print("LSTM前向传播 - 逐步数值计算")
print("=" * 70)
print()

# 简化参数
input_size = 2
hidden_size = 2

# 权重矩阵（简化：不使用拼接）
# 遗忘门
W_f = np.array([[0.5, 0.1], [0.2, 0.6]])
U_f = np.array([[0.3, 0.2], [0.1, 0.4]])
b_f = np.array([0.1, 0.0])

# 输入门
W_i = np.array([[0.4, 0.2], [0.1, 0.5]])
U_i = np.array([[0.2, 0.3], [0.4, 0.1]])
b_i = np.array([0.0, 0.1])

# 候选细胞状态
W_C = np.array([[0.6, 0.3], [0.2, 0.5]])
U_C = np.array([[0.3, 0.2], [0.2, 0.4]])
b_C = np.array([0.0, 0.0])

# 输出门
W_o = np.array([[0.3, 0.4], [0.2, 0.3]])
U_o = np.array([[0.4, 0.1], [0.3, 0.2]])
b_o = np.array([0.1, 0.0])

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# 输入序列
sequence = [
    np.array([1.0, 0.5]),
    np.array([0.5, 1.0]),
    np.array([1.0, 1.0]),
]

# 初始状态
h = np.zeros(hidden_size)
C = np.zeros(hidden_size)

print(f"初始状态: h_0 = {h}, C_0 = {C}")
print()

for t, x in enumerate(sequence):
    print(f"{'='*50}")
    print(f"时间步 t={t}: 输入 x_t = {x}")
    print(f"{'='*50}")
    
    # 1. 遗忘门
    f = sigmoid(W_f @ x + U_f @ h + b_f)
    print(f"1. 遗忘门: f_t = σ(W_f·x + U_f·h + b_f) = {np.round(f, 4)}")
    
    # 2. 输入门
    i = sigmoid(W_i @ x + U_i @ h + b_i)
    C_tilde = np.tanh(W_C @ x + U_C @ h + b_C)
    print(f"2. 输入门: i_t = {np.round(i, 4)}")
    print(f"   候选值: C̃_t = tanh(...) = {np.round(C_tilde, 4)}")
    
    # 3. 更新细胞状态
    C_new = f * C + i * C_tilde
    print(f"3. 更新细胞: C_t = f_t ⊙ C + i_t ⊙ C̃_t")
    print(f"   = {np.round(f, 4)} ⊙ {np.round(C, 4)} + {np.round(i, 4)} ⊙ {np.round(C_tilde, 4)}")
    print(f"   = {np.round(C_new, 4)}")
    
    # 4. 输出门
    o = sigmoid(W_o @ x + U_o @ h + b_o)
    h_new = o * np.tanh(C_new)
    print(f"4. 输出门: o_t = {np.round(o, 4)}")
    print(f"   h_t = o_t ⊙ tanh(C_t) = {np.round(h_new, 4)}")
    
    print()
    
    # 更新状态
    h = h_new
    C = C_new

print(f"{'='*50}")
print(f"最终状态:")
print(f"  h = {np.round(h, 4)}")
print(f"  C = {np.round(C, 4)}")

## 三、LSTM vs RNN：长距离依赖对比

让我们用同样的"长距离记忆"任务来对比LSTM和普通RNN的表现。

In [ ]:
# 对比实验: RNN vs LSTM 在长距离依赖任务上的表现
print("=" * 70)
print("对比实验: RNN vs LSTM 长距离记忆")
print("=" * 70)
print()

torch.manual_seed(42)

def generate_memory_data(n_samples, seq_len, n_features=5):
    X = torch.randn(n_samples, seq_len, n_features)
    signal = torch.randint(0, 2, (n_samples,)).float()
    X[:, 0, :] = signal.unsqueeze(1).expand(-1, n_features)
    y = signal
    return X, y

class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        _, h_n = self.rnn(x)
        return self.fc(h_n[-1]).squeeze(-1)

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.rnn = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        _, (h_n, _) = self.rnn(x)
        return self.fc(h_n[-1]).squeeze(-1)

# 测试不同序列长度
seq_lengths = [10, 20, 50, 100]
results = {'RNN': [], 'LSTM': []}

for seq_len in seq_lengths:
    print(f"\n序列长度: {seq_len}")
    print("-" * 40)
    
    X_train, y_train = generate_memory_data(500, seq_len)
    X_test, y_test = generate_memory_data(200, seq_len)
    
    for model_name, ModelClass in [('RNN', RNNModel), ('LSTM', LSTMModel)]:
        torch.manual_seed(42)
        model = ModelClass(input_size=5, hidden_size=32)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.01)
        
        best_acc = 0
        for epoch in range(100):
            model.train()
            optimizer.zero_grad()
            output = model(X_train)
            loss = criterion(output, y_train)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            model.eval()
            with torch.no_grad():
                preds = (model(X_test) > 0).float()
                acc = (preds == y_test).float().mean().item()
                best_acc = max(best_acc, acc)
        
        results[model_name].append(best_acc)
        print(f"  {model_name:4s}: 最佳准确率 = {best_acc:.2%}")

# 可视化结果
fig, ax = plt.subplots(figsize=(10, 6))

x_pos = np.arange(len(seq_lengths))
width = 0.35

bars1 = ax.bar(x_pos - width/2, results['RNN'], width, label='RNN', color='steelblue', alpha=0.8)
bars2 = ax.bar(x_pos + width/2, results['LSTM'], width, label='LSTM', color='coral', alpha=0.8)

ax.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='随机猜测')
ax.set_xlabel('序列长度', fontsize=12)
ax.set_ylabel('最佳测试准确率', fontsize=12)
ax.set_title('RNN vs LSTM: 长距离依赖任务', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels([str(s) for s in seq_lengths])
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# 在柱子上标数值
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{bar.get_height():.0%}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{bar.get_height():.0%}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# 打印结果汇总
print(f"\n{'序列长度':<10} {'RNN':>10} {'LSTM':>10}")
print("-" * 35)
for i, sl in enumerate(seq_lengths):
    print(f"{sl:<10} {results['RNN'][i]:>10.2%} {results['LSTM'][i]:>10.2%}")

## 四、GRU（门控循环单元）

### GRU vs LSTM

GRU是LSTM的**简化版**，核心区别：

| 特性 | LSTM | GRU |
|------|------|-----|
| 状态数 | 2个 (C_t, h_t) | 1个 (h_t) |
| 门数 | 3个 (遗忘、输入、输出) | 2个 (更新、重置) |
| 参数量 | 更多 | 更少 (约LSTM的75%) |
| 计算速度 | 较慢 | 较快 |
| 效果 | 略好 | 接近LSTM |

### GRU公式

**更新门 (Update Gate):**
$$z_t = \sigma(W_z \cdot [h_{t-1}, x_t] + b_z)$$
控制保留多少旧信息和接受多少新信息。

**重置门 (Reset Gate):**
$$r_t = \sigma(W_r \cdot [h_{t-1}, x_t] + b_r)$$
控制遗忘多少过去的信息。

**候选隐藏状态:**
$$\tilde{h}_t = \tanh(W_h \cdot [r_t \odot h_{t-1}, x_t] + b_h)$$

**最终隐藏状态:**
$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

### GRU的直觉

- 当 $z_t \approx 1$：$h_t \approx \tilde{h}_t$，完全接受新信息
- 当 $z_t \approx 0$：$h_t \approx h_{t-1}$，完全保留旧信息
- 和LSTM一样，当 $z_t \approx 0$ 时，梯度可以近似直接传播到 $h_{t-1}$

In [ ]:
# GRU可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LSTM示意图
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title('LSTM结构\n(2个状态, 3个门)', fontsize=12, fontweight='bold')

# 细胞状态线
ax.plot([1, 9], [6.5, 6.5], 'k-', linewidth=3)
ax.text(0.3, 6.5, 'C', fontsize=14, ha='center', va='center')

# 隐藏状态线
ax.plot([1, 9], [1.5, 1.5], 'k-', linewidth=3)
ax.text(0.3, 1.5, 'h', fontsize=14, ha='center', va='center')

# 三个门
ax.text(3, 4.5, '遗忘门', fontsize=10, ha='center',
        bbox=dict(boxstyle='round', facecolor='#FFD700', alpha=0.7))
ax.text(5, 4.5, '输入门', fontsize=10, ha='center',
        bbox=dict(boxstyle='round', facecolor='#FFD700', alpha=0.7))
ax.text(7, 4.5, '输出门', fontsize=10, ha='center',
        bbox=dict(boxstyle='round', facecolor='#FFD700', alpha=0.7))

# 操作符
ax.text(3, 6.5, '×', fontsize=16, ha='center', va='center', color='red', fontweight='bold')
ax.text(5, 6.5, '+', fontsize=16, ha='center', va='center', color='green', fontweight='bold')
ax.text(7, 2.0, '×', fontsize=16, ha='center', va='center', color='red', fontweight='bold')

# GRU示意图
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title('GRU结构\n(1个状态, 2个门)', fontsize=12, fontweight='bold')

# 隐藏状态线
ax.plot([1, 9], [4.0, 4.0], 'k-', linewidth=3)
ax.text(0.3, 4.0, 'h', fontsize=14, ha='center', va='center')

# 两个门
ax.text(3.5, 5.5, '更新门 z_t', fontsize=10, ha='center',
        bbox=dict(boxstyle='round', facecolor='#FFD700', alpha=0.7))
ax.text(6.5, 5.5, '重置门 r_t', fontsize=10, ha='center',
        bbox=dict(boxstyle='round', facecolor='#FFD700', alpha=0.7))

# 候选状态
ax.text(5, 2.5, 'h̃_t (候选)', fontsize=10, ha='center',
        bbox=dict(boxstyle='round', facecolor='#87CEEB', alpha=0.7))

# 操作符
ax.text(5, 4.0, '×(1-z)+z×h̃', fontsize=11, ha='center', va='center', color='red', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# GRU vs LSTM 实验对比
print("\n=== GRU vs LSTM 实验对比 ===")
print()

class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.rnn = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        _, h_n = self.rnn(x)
        return self.fc(h_n[-1]).squeeze(-1)

seq_len = 50
print(f"序列长度: {seq_len}")
print("-" * 50)

X_train, y_train = generate_memory_data(500, seq_len)
X_test, y_test = generate_memory_data(200, seq_len)

for model_name, ModelClass in [('LSTM', LSTMModel), ('GRU', GRUModel)]:
    import time
    torch.manual_seed(42)
    model = ModelClass(input_size=5, hidden_size=32)
    n_params = sum(p.numel() for p in model.parameters())
    
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    
    start_time = time.time()
    best_acc = 0
    for epoch in range(100):
        model.train()
        optimizer.zero_grad()
        output = model(X_train)
        loss = criterion(output, y_train)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        model.eval()
        with torch.no_grad():
            preds = (model(X_test) > 0).float()
            acc = (preds == y_test).float().mean().item()
            best_acc = max(best_acc, acc)
    train_time = time.time() - start_time
    
    print(f"{model_name:4s}: 准确率={best_acc:.2%}, 参数量={n_params}, 训练时间={train_time:.2f}s")

## 五、本章总结

### 核心要点

1. **LSTM通过细胞状态+C_t解决梯度消失**
   - 细胞状态的更新是加法：$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$
   - 当遗忘门 $f_t \approx 1$ 时，梯度可以无损传播

2. **三个门的作用**
   - 遗忘门：控制保留多少旧记忆
   - 输入门：控制接受多少新记忆
   - 输出门：控制输出多少当前状态

3. **GRU是LSTM的简化版**
   - 参数量少75%，训练更快
   - 效果接近LSTM，适合数据量不大的场景

4. **实验验证：LSTM/GRU能处理50+步的长距离依赖，而RNN只能处理10步左右**

### 下一篇预告

下一篇将学习**Seq2Seq架构和注意力机制**——这是Transformer的前身。